In [1]:
from bs4 import BeautifulSoup
import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
import time


async def get_html(url, selector, sleep=5, retries=5):
    html = None
    for i in range(1, retries + 1):
        time.sleep(sleep * i)
        try:
            async with async_playwright() as p:
                browser = await p.webkit.launch()
                page = await browser.new_page()
                await page.goto(url)
                print(await page.title())
                html = await page.inner_html(selector)
        except PlaywrightTimeout:
            print(f"Timeout error on {url}")
            continue
        else:
            break
    return html

In [2]:
html = await get_html(
    "https://finance.yahoo.com/markets/stocks/most-active/", ".tableContainer"
)

Most Active Stocks: US stocks with the highest trading volume today - Yahoo Finance


In [3]:
soup = BeautifulSoup(html)
stocks = pd.read_html(str(soup))[0]

/var/folders/mg/y6t9hdxs67v8yvwhckq3gj4h0000gn/T/ipykernel_57022/1317362581.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  stocks = pd.read_html(str(soup))[0]


In [4]:
stocks = stocks.dropna(axis=0, how="all")
stocks = stocks.dropna(axis=1, how="all")
stocks = stocks.drop("52 Wk Range", axis=1)

In [5]:
def extract_price(all_prices):
    price = str(all_prices).split(" ")[0]
    return price


def format_numbers(num):
    num = str(num)
    if num.endswith("%"):
        num = num.split("%")[0]
        if num.startswith("+"):
            num = num.split("+")[-1]
    elif num.endswith("M"):
        num = float(num.split("M")[0]) * 1e6
    elif num.endswith("B"):
        num = float(num.split("B")[0]) * 1e9
    elif num.endswith("T"):
        num = float(num.split("T")[0]) * 1e12
    return num

stocks["Price"] = stocks["Price"].apply(extract_price)

In [6]:
stocks["Change %"] = stocks["Change %"].apply(format_numbers)
stocks["Volume"] = stocks["Volume"].apply(format_numbers)
stocks["Avg Vol (3M)"] = stocks["Avg Vol (3M)"].apply(format_numbers)
stocks["Market Cap"] = stocks["Market Cap"].apply(format_numbers)
stocks["52 Wk Change %"] = stocks["52 Wk Change %"].apply(format_numbers)
stocks["P/E Ratio (TTM)"] = stocks["P/E Ratio (TTM)"].replace(to_replace="--", value=0)

In [7]:
stocks

,Symbol,Name,Price,Change,Change %,Volume,Avg Vol (3M),Market Cap,P/E Ratio (TTM),52 Wk Change %
0,NVDA,NVIDIA Corporation,177.88,-5.46,-2.98,176436000.0,176387000.0,4.323000e+12,37.36,62.69
1,ONDS,Ondas Inc.,9.83,-0.63,-6.05,154048000.0,92534000.0,4.420000e+09,0,"1,196.66"
2,AAL,American Airlines Group Inc.,11.18,-0.61,-5.17,97585000.0,58513000.0,7.376000e+09,74.08,-9.52
3,MRVL,"Marvell Technology, Inc.",89.57,13.89,18.35,84102000.0,14771000.0,7.820000e+10,27.47,6.83
4,BBD,Banco Bradesco S.A.,3.68,-0.06,-1.60,80864000.0,37590000.0,3.915600e+10,9.16,80.68
5,DAWN,"Day One Biopharmaceuticals, Inc.",21.20,8.42,65.88,77304000.0,2739000.0,2.190000e+09,0,47.40
6,PLUG,Plug Power Inc.,2.13,-0.16,-6.99,72550000.0,100328000.0,2.964000e+09,0,28.65
7,INTC,Intel Corporation,43.42,-2.53,-5.51,72224000.0,101129000.0,2.168910e+11,0,122.63
8,SOFI,"SoFi Technologies, Inc.",18.91,-0.34,-1.79,71885000.0,56348000.0,2.410900e+10,48.49,52.90
9,PLTR,Palantir Technologies Inc.,157.13,4.46,2.92,69128000.0,47539000.0,3.758160e+11,243.16,79.80
